# 10장. 회귀분석(2)미세먼지 회귀분석-대기오염데이터 수집 크롤링

### 공공데이터포털에서 제공하는 오픈API 사용
#### https://www.data.go.kr/data/15073855/openapi.do#tab_layer_detail_function

In [2]:
import urllib.request
import datetime
import json
import pandas as pd

In [3]:
# 공공데이터포탈 data.go.kr에서 발급받은 인증키를 복사하여 사용

ServiceKey= "13931bef0220037497fe74a7c910b63e57de39b2d23ba5eeae6ef8a0783cebd6"

In [4]:
#[CODE 1]
def getRequestUrl(url):    
    req = urllib.request.Request(url)    
    try: 
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print ("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for URL : %s" % (datetime.datetime.now(), url))
        return None

In [5]:
#[CODE 2]
def reqAirInfo(where, beginDay, endDay, numOfRows):    
    service_url = 'http://apis.data.go.kr/B552584/ArpltnStatsSvc/getMsrstnAcctoRDyrg'
    parameters = "?returnType=json&serviceKey=" + ServiceKey   #인증키
    parameters += "&msrstnName=" + (urllib.parse.quote(where))
    parameters += "&inqBginDt=" + beginDay
    parameters += "&inqEndDt=" + endDay
    parameters += "&numOfRows=" + numOfRows

    url = service_url + parameters
    print(url) #액세스 거부 여부 확인용 출력    
    
    responseDecode = getRequestUrl(url)   #[CODE 1]
    
    if (responseDecode == None):
        return None
    else:
         return json.loads(responseDecode)

In [6]:
#[CODE 3]
def getAirInfoItem(item, result):  
    msrstnName = item['msrstnName']  #측정소
    msurDt = item['msurDt']  #측정일
    so2Value = item['so2Value'] #아황산가스 평균농도
    coValue = item['coValue']  #일산화탄소 평균농도
    o3Value = item['o3Value']  #오존 평균농도
    no2Value = item['no2Value'] #이산화질소 평균농도
    pm10Value = item['pm10Value'] #미세먼지 평균농도
    pm25Value = item['pm25Value'] #초미세먼지 평균농도   
    
    result.append([msrstnName, msurDt, so2Value, coValue, o3Value, no2Value, pm10Value, pm25Value])

In [8]:
where = input('Enter the measurement station for air pollution data collection: ')
beginDay = input('Enter the start date for air pollution data collection (YYYYMMDD): ')
endDay = input('Enter the end date for air pollution data collection (YYYYMMDD): ')

In [9]:
#url parameter로 추가할 numOfRows 값 만들기
b_date = datetime.datetime.strptime(beginDay, '%Y%m%d')
e_date = datetime.datetime.strptime(endDay, '%Y%m%d')
days = e_date - b_date
numOfRows = str(days.days)  #크롤링 데이터 수 <= 수집 기간의 일수

jsonResponse = [] #크롤링한 JSON 데이터를 저장할 리스트
result = []       #JSON에서 항목을 추출하여 저장할 리스트 -> csv로 저장

In [10]:
#데이터 요청에 대한 응답 데이터를 저장한 json 객체를 반환받아 저장하기
jsonResponse = reqAirInfo(where, beginDay, endDay, numOfRows)  #[CODE 2] 

http://apis.data.go.kr/B552584/ArpltnStatsSvc/getMsrstnAcctoRDyrg?returnType=json&serviceKey=13931bef0220037497fe74a7c910b63e57de39b2d23ba5eeae6ef8a0783cebd6&msrstnName=tashkent&inqBginDt=20240609&inqEndDt=20240909&numOfRows=92
[2026-04-29 09:20:59.514641] Url Request Success


In [11]:
#JSON 데이터(jsonResponse)에서 데이터 하나(일 측정 데이터)씩 정리하여 리스트 result에 저장
for item in jsonResponse['response']['body']['items']:            
    getAirInfoItem(item, result)  #[CODE 3]       

In [14]:
#파일저장 : csv 파일   
columnNames = ["location", "day", "so2", "co", "o3", "no2", "pm10", "pm25"]
result_df = pd.DataFrame(result, columns = columnNames)
result_df.to_csv('../10장_data/대기오염데이터_%s_%s_%s.csv' % (where, beginDay, endDay), index=False, encoding='CP949')
      
print ('./10장_data/대기오염데이터_%s_%s_%s.csv 저장완료.' % (where, beginDay, endDay))

./10장_data/대기오염데이터_tashkent_20240609_20240909.csv 저장완료.
